In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import struct
import os
import zipfile

def load_idx(path, is_image):
    with open(path, "rb") as fh:
        if is_image:
            _, n, r, c = struct.unpack(">IIII", fh.read(16))
            buf = np.frombuffer(fh.read(), np.uint8)
            return buf.reshape(n, r * c).astype(np.float32) / 255.0, (r, c)
        else:
            _, n = struct.unpack(">II", fh.read(8))
            return np.frombuffer(fh.read(), np.uint8), None


x_train, (H, W) = load_idx("train-images-idx3-ubyte/train-images-idx3-ubyte", True)
y_train = load_idx("train-labels-idx1-ubyte/train-labels-idx1-ubyte", False)[0]

x_test, _ = load_idx("t10k-images-idx3-ubyte/t10k-images-idx3-ubyte", True)
y_test = load_idx("t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte", False)[0]

# One-hot нужен потому, что выход softmax - это распределение вероятностей, и сравнивать его нужно с распределением, а не с числом (в y - только метки чисел)
# Пример: Метка 3 → вектор [0,0,0,1,0,0,0,0,0,0] (единица на позиции 3)
def one_hot(y):
    out = np.zeros((y.size, 10), dtype=np.float32)
    out[np.arange(y.size), y] = 1.0
    return out

y_train_oh = one_hot(y_train)

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape,  y_test.shape)
print("Форма изображения:", (28, 28))

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"label={y_train[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

# Инициализация весов для компенсации потери сигнала на ReLU
def init_params(n_in=784, n_hidden=300, n_out=10):
    rng = np.random.default_rng(42)
    W1 = rng.normal(0, np.sqrt(2.0 / n_in), (n_hidden, n_in )).astype(np.float32) # 300 скрытых нейронов на 784 входа 
    b1 = np.zeros(n_hidden, dtype=np.float32) # смещение каждого скрытого нейрона
    W2 = rng.normal(0, np.sqrt(2.0 / n_hidden), (n_out,  n_hidden)).astype(np.float32) # 10 выходов на 300 скрытых 
    b2 = np.zeros(n_out, dtype=np.float32) # смещение каждого класса

    return W1, b1, W2, b2

# Прямой проход: вычисление выхода сети по входу
def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1.T + b1 # комбинация входов для каждого скрытого нейрона (W1.T - копия W1, чтобы не потерять данные)
    H = np.maximum(Z1, 0.0) # применение ReLU по элементно, если сигнал отрицательный - выключить нейрон, если положительный - пропустить
    Z2 = H @ W2.T + b2 # сырое число выходного слоя по одному числу на класс
    Z2 = Z2 - Z2.max(axis=1, keepdims=True) # для стабильности softmax (вычитаем максимум по каждой строке)
    expZ = np.exp(Z2) # поэлементное возведение в экспоненту
    Yhat = expZ / expZ.sum(axis=1, keepdims=True) # нормирование expZ, чтобы сумма по каждой строке была равна 1

    return Z1, H, Yhat

# Обратный проход: по известному выходу сети и правильному ответу вычислить, как изменить каждый параметр, чтобы ошибка уменьшилась
def backward(X, Y_oh, Z1, H, Yhat, W2):
    B = X.shape[0] # размер батча

    dZ2 = (Yhat - Y_oh) / B # градиент по логитам выхода
    dW2 = dZ2.T @ H # градиент по весам выхода
    db2 = dZ2.sum(axis=0) # градиент по смещению выхода

    dH  = dZ2 @ W2 # обратное распространение на скрытый слой
    dZ1 = dH * (Z1 > 0) # градиент через ReLU
    dW1 = dZ1.T @ X # градиент по весам скрытого слоя
    db1 = dZ1.sum(axis=0) # градиент по смещению скрытого слоя

    return dW1, db1, dW2, db2

# Стохастический градиентный спуск по мини-батчам
def train(X, Y, Y_oh, epochs=20, batch_size=64, lr=0.1,
          n_hidden=300, X_test=None, y_test=None):
    W1, b1, W2, b2 = init_params(n_hidden=n_hidden)
    N = X.shape[0]

    for epoch in range(epochs):
        t0 = time.time() # время начала эпохи
        idx = np.random.permutation(N)
        X_sh, Y_oh_sh = X[idx], Y_oh[idx]

        for start in range(0, N, batch_size):
            end = start + batch_size
            xb = X_sh[start:end]
            yb = Y_oh_sh[start:end]

            Z1, H, Yhat = forward(xb, W1, b1, W2, b2)
            dW1, db1, dW2, db2 = backward(xb, yb, Z1, H, Yhat, W2)

            W1 -= lr * dW1;  b1 -= lr * db1
            W2 -= lr * dW2;  b2 -= lr * db2

        _, _, Yhat_tr = forward(X, W1, b1, W2, b2)
    
        y_pred = Yhat_tr.argmax(axis=1)
        y_true = Y.ravel()
        train_err = 1.0 - np.mean(y_pred == y_true)

        dt = time.time() - t0
        print(f"Epoch {epoch+1:2d}/{epochs} | train error = {train_err:.4f} | {dt:.1f}s")

    _, _, Yhat_te = forward(X_test, W1, b1, W2, b2)
    test_acc = np.mean(Yhat_te.argmax(1) == y_test)
    test_err = 1.0 - test_acc
    print(f"\nTest error = {test_err:.4f}, Test accuracy = {test_acc:.4f}")

train(
    x_train, y_train, y_train_oh,
    epochs=20, batch_size=64, lr=0.1, n_hidden=300,
    X_test=x_test, y_test=y_test)

FileNotFoundError: [Errno 2] No such file or directory: 'train-images-idx3-ubyte/train-images-idx3-ubyte'